<a href="https://colab.research.google.com/github/arnavm2k3/TorchCode-Solutions/blob/main/Architecture_Adaptation/28_moe.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🔴 Hard: Mixture of Experts (MoE)

Implement a **Mixture of Experts** layer (Mixtral / Switch Transformer style).

### Signature
```python
class MixtureOfExperts(nn.Module):
    def __init__(self, d_model, d_ff, num_experts, top_k=2): ...
    def forward(self, x: Tensor) -> Tensor:
        # x: (B, S, D) -> (B, S, D)
```

### Architecture
- `self.router`: `nn.Linear(d_model, num_experts)` — gating network
- `self.experts`: `nn.ModuleList` of MLPs `(Linear→ReLU→Linear)`
- For each token: select top-k experts, compute weighted sum of their outputs

### References:
Mixture of Experts Explained - [Link](https://huggingface.co/blog/moe)

In [1]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.5/48.5 kB 3.0 MB/s eta 0:00:00


In [2]:
import torch
import torch.nn as nn

In [11]:
# ✏️ YOUR IMPLEMENTATION HERE

class MixtureOfExperts(nn.Module):
    def __init__(self, d_model, d_ff, num_experts, top_k=2):
        super().__init__()
        # router + experts
        self.router = nn.Linear(d_model, num_experts)
        self.experts = nn.ModuleList([
            nn.Sequential(
                nn.Linear(d_model, d_ff),
                nn.ReLU(),
                nn.Linear(d_ff, d_model)
                ) for _ in range(num_experts)
        ])
        self.top_k = top_k

    def forward(self, x):
        # route tokens to top-k experts
        top_k_scores, top_k_idx = self.router(x).topk(self.top_k, dim=-1)
        top_k_scores = torch.softmax(top_k_scores, dim=-1)



In [12]:
# 🧪 Debug
moe = MixtureOfExperts(32, 64, num_experts=4, top_k=2)
x = torch.randn(2, 8, 32)
print('Output:', moe(x).shape)
print('Params:', sum(p.numel() for p in moe.parameters()))

vals: tensor([[[ 8.8556e-01,  4.0162e-01],
         [ 8.4698e-01,  4.7778e-01],
         [ 3.9304e-01, -1.2533e-02],
         [ 5.3235e-01,  4.7086e-01],
         [ 1.5901e+00,  4.9903e-01],
         [ 1.4499e+00,  4.5138e-01],
         [ 2.4324e-01,  1.6030e-01],
         [ 4.6737e-01,  1.6531e-01]],

        [[ 6.1370e-01,  3.5420e-01],
         [ 9.0012e-01,  4.0596e-01],
         [ 1.0561e+00,  2.4667e-01],
         [ 7.9511e-01,  9.0950e-02],
         [ 2.9340e-01, -3.2071e-04],
         [ 5.2140e-01, -1.4284e-01],
         [ 4.2578e-01,  3.8871e-01],
         [ 6.2677e-01, -4.6847e-02]]], grad_fn=<TopkBackward0>)
idx:tensor([[[1, 2],
         [3, 1],
         [1, 0],
         [0, 2],
         [0, 1],
         [0, 1],
         [1, 3],
         [1, 2]],

        [[2, 1],
         [0, 1],
         [0, 2],
         [1, 0],
         [1, 2],
         [1, 0],
         [2, 1],
         [1, 2]]])


AttributeError: 'NoneType' object has no attribute 'shape'

In [ ]:
# ✅ SUBMIT
from torch_judge import check
check('moe')